# Customer Support Ticket Intelligence Platform
## Phase 1: Project Planning, Dataset Inspection & Data Preparation

This notebook guides us through **Phase 1** of our project. As NLP Researchers and Machine Learning Engineers, our goal is to investigate the customer support tickets dataset, inspect its statistical structures, establish the business case, and produce a **clean, deduplicated, reproducible modeling dataset** before building deep learning sequence models.

> **Columns used**: `issue_description` (free-text input) · `category` (multi-class classification target)  
> **Raw size**: ~1.9 million rows  
> **Objective**: Automated ticket routing — predict `category` from `issue_description`

### 🏢 Business Case & Operational ROI Formulation

We want to automate the routing of customer support tickets to the correct team. Operational math:
- **Weekly Volume**: 10,000 support tickets
- **Manual Triaging Time**: 2 minutes per ticket
- **Cost**: $0.40/minute ($24/hour agent rate)
- **Goal**: Build a classifier predicting `category` from `issue_description`. Auto-routing tickets above a confidence threshold (e.g. $\tau = 80\%$) reduces manual overhead, saves hundreds of analyst-hours per week, and cuts time-to-route from hours to milliseconds.

---
### 1. Load Dataset & Initial Profiling

Before any cleaning, we profile the **raw** dataset to understand its scale, class structure, and the extent of duplication.

In [22]:
import pandas as pd
import numpy as np

# ── Load raw data ────────────────────────────────────────────────────────────
data_path = '../data/consumer-complaint-database.csv'
df = pd.read_csv(data_path, low_memory=False)
print(len(df))

# Standardise column names to match our NLP pipeline schema
df = df.rename(columns={
    'Consumer complaint narrative': 'issue_description',
    'Product': 'category'
})

# Keep only the two columns required by the NLP pipeline
df = df[['issue_description', 'category']].copy()

# Drop rows where either column is null (narrative must exist to be useful)
df = df.dropna(subset=['issue_description', 'category']).reset_index(drop=True)

print(f"Raw dataset shape (after null drop): {df.shape}")
print(f"Columns: {list(df.columns)}")

1282355
Raw dataset shape (after null drop): (383564, 2)
Columns: ['issue_description', 'category']


---
### 2. Deduplication & Stratified Sampling Pipeline

#### Why this matters
The raw dataset contains **exact duplicate (text, label) pairs** — the same ticket text assigned to the same category. Keeping duplicates causes:
- **Data leakage**: identical samples in train and validation sets → optimistically inflated metrics.
- **Memorisation bias**: the model learns to recall strings rather than generalise from semantics.
- **Wasted compute**: training on redundant rows adds epochs without adding information.

#### Why 200,000 rows?
For an NLP portfolio project using sequence models (LSTM, Transformer):
- **Minimum for generalisation**: ~10,000–20,000 samples per class is sufficient for strong embeddings.
- **Memory budget**: a 200K row dataset with tokenised sequences (max length 128) fits comfortably in RAM and trains in minutes on a modern GPU.
- **Statistical representativeness**: 200K rows is large enough that per-class proportions are preserved to <0.1% error, giving a fair evaluation baseline.
- **Practical ceiling**: beyond ~500K rows, incremental gains diminish for fixed-vocabulary bag-of-words and LSTM models; compute cost grows linearly while accuracy gains are sub-linear.

In [21]:
# ════════════════════════════════════════════════════════════════════════════
# STEP A: RAW DATASET ANALYSIS
# ════════════════════════════════════════════════════════════════════════════

total_rows_raw         = len(df)
unique_texts_raw       = df['issue_description'].nunique()
n_categories_raw       = df['category'].nunique()
class_dist_raw         = df['category'].value_counts()
n_duplicate_pairs_raw  = total_rows_raw - df.drop_duplicates(
                             subset=['issue_description', 'category']
                         ).shape[0]
pct_duplicates_raw     = (n_duplicate_pairs_raw / total_rows_raw) * 100

print("═" * 55)
print("  RAW DATASET PROFILE")
print("═" * 55)
print(f"  Total rows             : {total_rows_raw:>12,}")
print(f"  Unique ticket texts    : {unique_texts_raw:>12,}")
print(f"  Number of categories   : {n_categories_raw:>12,}")
print(f"  Duplicate (text,label) : {n_duplicate_pairs_raw:>12,}  ({pct_duplicates_raw:.2f}%)")
print("\n  Class Distribution (counts):")
for cat, cnt in class_dist_raw.items():
    pct = cnt / total_rows_raw * 100
    print(f"    {cat:<45} {cnt:>8,}  ({pct:5.2f}%)")

═══════════════════════════════════════════════════════
  RAW DATASET PROFILE
═══════════════════════════════════════════════════════
  Total rows             :      383,564
  Unique ticket texts    :      366,945
  Number of categories   :           18
  Duplicate (text,label) :       16,441  (4.29%)

  Class Distribution (counts):
    Credit reporting, credit repair services, or other personal consumer reports   92,378  (24.08%)
    Debt collection                                 86,710  (22.61%)
    Mortgage                                        52,987  (13.81%)
    Credit reporting                                31,588  ( 8.24%)
    Student loan                                    21,810  ( 5.69%)
    Credit card or prepaid card                     21,379  ( 5.57%)
    Credit card                                     18,838  ( 4.91%)
    Bank account or service                         14,885  ( 3.88%)
    Checking or savings account                     12,881  ( 3.36%)
    Consumer 

In [23]:
# ════════════════════════════════════════════════════════════════════════════
# STEP B: DEDUPLICATION
#   Remove exact (issue_description, category) duplicate pairs.
#   We keep the first occurrence so the result is deterministic.
# ════════════════════════════════════════════════════════════════════════════

df_clean = (
    df
    .drop_duplicates(subset=['issue_description', 'category'], keep='first')
    .reset_index(drop=True)
)

rows_removed = total_rows_raw - len(df_clean)
print(f"Rows removed by deduplication : {rows_removed:,}")
print(f"df_clean shape                : {df_clean.shape}")

Rows removed by deduplication : 16,441
df_clean shape                : (367123, 2)


In [31]:
# ════════════════════════════════════════════════════════════════════════════
# STEP C: AUTOMATIC SAMPLE SIZE DECISION
#   · If deduplicated dataset < 250,000 rows  → keep all rows (df_sample = df_clean)
#   · Otherwise                               → stratified sample of ~200,000 rows
# ════════════════════════════════════════════════════════════════════════════

DEDUP_THRESHOLD = 250_000   # rows: below this, keep everything
TARGET_SAMPLE   = 200_000   # rows: target size when sampling is needed
RANDOM_STATE    = 42        # reproducibility seed

if len(df_clean) < DEDUP_THRESHOLD:
    # ── Dataset is small enough: no further sampling needed ─────────────────
    df_sample = df_clean.copy()
    print(f"Deduplicated rows ({len(df_clean):,}) < {DEDUP_THRESHOLD:,} threshold.")
    print("→ Keeping full deduplicated dataset (df_sample = df_clean).")

else:
    # ── Dataset is large: stratified sample to ~TARGET_SAMPLE rows ──────────
    # Compute per-class sample counts proportional to class frequency.
    # Using a fractional approach (GroupBy + sample) is robust to rare classes.

    sampling_fraction = TARGET_SAMPLE / len(df_clean)

    df_sample = (
        df_clean
        .groupby('category', group_keys=True)          # iterate over each class
        .apply(
            lambda grp: grp.sample(
                n=max(1, round(len(grp) * sampling_fraction)),  # ≥1 per class
                random_state=RANDOM_STATE
            )
        )
        .reset_index(level=0)                           # retrieve the category column from index
        .sample(frac=1, random_state=RANDOM_STATE)      # global shuffle
        .reset_index(drop=True)
    )
    df_sample = df_sample[['issue_description', 'category']]

    print(f"Deduplicated rows ({len(df_clean):,}) ≥ {DEDUP_THRESHOLD:,} threshold.")
    print(f"→ Stratified sample created: {len(df_sample):,} rows.")
    print(df_sample.head())

Deduplicated rows (367,123) ≥ 250,000 threshold.
→ Stratified sample created: 200,000 rows.
                                   issue_description  \
0  I paid my debt and when i was getting the amou...   
1  I have had an ongoing problem and now will res...   
2  Complaint Details I filled out an application ...   
3  I recently applied for seasonal work for XXXX ...   
4  XXXX XXXX XXXX XXXX of XXXX VA XXXX was servic...   

                                            category  
0                                    Debt collection  
1  Credit reporting, credit repair services, or o...  
2                                           Mortgage  
3  Credit reporting, credit repair services, or o...  
4                                        Credit card  


In [32]:
# ════════════════════════════════════════════════════════════════════════════
# STEP D: BEFORE / AFTER STATISTICS
# ════════════════════════════════════════════════════════════════════════════

def _profile(frame: pd.DataFrame, label: str) -> dict:
    """Return a summary dict for a given dataframe."""
    n_rows        = len(frame)
    n_unique_text = frame['issue_description'].nunique()
    n_cats        = frame['category'].nunique()
    n_dupe_pairs  = n_rows - frame.drop_duplicates(
                        subset=['issue_description', 'category']).shape[0]
    pct_dupe      = (n_dupe_pairs / n_rows * 100) if n_rows else 0.0
    return {
        'Dataset'            : label,
        'Rows'               : n_rows,
        'Unique Texts'        : n_unique_text,
        'Categories'         : n_cats,
        'Duplicate Pairs'    : n_dupe_pairs,
        'Duplicate %'        : round(pct_dupe, 4),
    }

stats = pd.DataFrame([
    _profile(df,        'raw (post null-drop)'),
    _profile(df_clean,  'df_clean (deduplicated)'),
    _profile(df_sample, 'df_sample (final modeling)'),
]).set_index('Dataset')

print("\n" + "═" * 70)
print("  BEFORE / AFTER STATISTICS")
print("═" * 70)
print(stats.to_string())

print("\n" + "═" * 70)
print("  df_sample — CLASS DISTRIBUTION")
print("═" * 70)
class_dist_sample = df_sample['category'].value_counts()
for cat, cnt in class_dist_sample.items():
    pct = cnt / len(df_sample) * 100
    bar = '█' * int(pct / 1)    # 1 block per 1%
    print(f"  {cat:<45} {cnt:>7,}  ({pct:5.2f}%) {bar}")


══════════════════════════════════════════════════════════════════════
  BEFORE / AFTER STATISTICS
══════════════════════════════════════════════════════════════════════
                              Rows  Unique Texts  Categories  Duplicate Pairs  Duplicate %
Dataset                                                                                   
raw (post null-drop)        383564        366945          18            16441       4.2864
df_clean (deduplicated)     367123        366945          18                0       0.0000
df_sample (final modeling)  200000        199942          18                0       0.0000

══════════════════════════════════════════════════════════════════════
  df_sample — CLASS DISTRIBUTION
══════════════════════════════════════════════════════════════════════
  Debt collection                                45,934  (22.97%) ██████████████████████
  Credit reporting, credit repair services, or other personal consumer reports  43,836  (21.92%) ████████████

In [33]:
# ════════════════════════════════════════════════════════════════════════════
# STEP E: PERSIST OUTPUTS
#   · df_clean  → data/df_clean.csv         (deduplicated, full)
#   · df_sample → data/df_sample.csv        (final modeling dataset)
# ════════════════════════════════════════════════════════════════════════════

clean_path  = '../data/df_clean.csv'
sample_path = '../data/df_sample.csv'

df_clean.to_csv(clean_path,  index=False)
df_sample.to_csv(sample_path, index=False)

print(f"df_clean  saved → {clean_path}   ({len(df_clean):,} rows)")
print(f"df_sample saved → {sample_path}  ({len(df_sample):,} rows)")

# Quick sanity check on the final dataset
print("\ndf_sample preview:")
df_sample.head(3)

df_clean  saved → ../data/df_clean.csv   (367,123 rows)
df_sample saved → ../data/df_sample.csv  (200,000 rows)

df_sample preview:


,issue_description,category
0,I paid my debt and when i was getting the amou...,Debt collection
1,I have had an ongoing problem and now will res...,"Credit reporting, credit repair services, or o..."
2,Complaint Details I filled out an application ...,Mortgage


---
### 3. Missing Value Analysis

Audit the **final modeling dataset** (`df_sample`) for any remaining nulls.

In [34]:
missing_vals = df_sample.isnull().sum()
missing_pcts = (missing_vals / len(df_sample)) * 100
missing_df   = pd.DataFrame({
    'Missing Counts': missing_vals,
    'Percentage (%)': missing_pcts.round(4)
})

result = missing_df[missing_df['Missing Counts'] > 0]
if result.empty:
    print("✅  No missing values in df_sample — dataset is clean.")
else:
    print(result)

✅  No missing values in df_sample — dataset is clean.


---
### 4. Target Distribution (`category`)

Our **NLP target** is `category`. We verify proportions are preserved after stratified sampling.

In [35]:
print("=== Category Distribution in df_sample ===")
cat_dist = df_sample['category'].value_counts(normalize=True).mul(100).round(3)
print(cat_dist.to_string())

# Verify: proportion drift vs raw
raw_dist    = df['category'].value_counts(normalize=True).mul(100)
sample_dist = df_sample['category'].value_counts(normalize=True).mul(100)
drift       = (sample_dist - raw_dist).abs().dropna()

print(f"\nMax proportion drift (raw vs sample): {drift.max():.4f} percentage points")
print("(Values near 0 confirm stratification preserved class balance.)") 

=== Category Distribution in df_sample ===
category
Debt collection                                                                 22.967
Credit reporting, credit repair services, or other personal consumer reports    21.918
Mortgage                                                                        14.422
Credit reporting                                                                 8.124
Student loan                                                                     5.933
Credit card or prepaid card                                                      5.800
Credit card                                                                      5.109
Bank account or service                                                          4.046
Checking or savings account                                                      3.502
Consumer Loan                                                                    2.572
Vehicle loan or lease                                                         

---
### 5. Text Length Characteristics

We analyse `issue_description` — this is the raw text our NLP models will consume.

In [36]:
# Tokenize using basic whitespace split
df_sample = df_sample.copy()   # avoid SettingWithCopyWarning
df_sample['char_length'] = df_sample['issue_description'].astype(str).apply(len)
df_sample['word_length'] = df_sample['issue_description'].astype(str).apply(lambda x: len(x.split()))

print("Text characteristics (Characters):")
print(df_sample['char_length'].describe().round(2), '\n')
print('Text characteristics (Words):')
print(df_sample['word_length'].describe().round(2))

Text characteristics (Characters):
count    200000.00
mean       1115.21
std        1203.41
min           5.00
25%         412.00
50%         774.00
75%        1405.00
max       31735.00
Name: char_length, dtype: float64 

Text characteristics (Words):
count    200000.00
mean        202.92
std         216.44
min           1.00
25%          75.00
50%         141.00
75%         256.00
max        6314.00
Name: word_length, dtype: float64


---
### 6. Inspecting Text Patterns & Vocabulary Size

In [37]:
from collections import Counter
import re

# ── Vocabulary size ──────────────────────────────────────────────────────────
all_words = []
for text in df_sample['issue_description']:
    tokens = re.findall(r'\b[a-z]+\b', str(text).lower())
    all_words.extend(tokens)

word_freq = Counter(all_words)
print(f"Total tokens       : {len(all_words):,}")
print(f"Unique vocabulary  : {len(word_freq):,}")
print(f"\nTop-20 tokens:")
for word, cnt in word_freq.most_common(20):
    print(f"  {word:<25} {cnt:>8,}")

# ── Sample texts per category ────────────────────────────────────────────────
print("\n" + "=" * 65)
print("Sample texts (one per category):")
print("=" * 65)
for cat in df_sample['category'].unique()[:5]:
    sample_text = df_sample[df_sample['category'] == cat]['issue_description'].iloc[0]
    print(f"\n[{cat}]\n{str(sample_text)[:300]}...")

Total tokens       : 40,043,478
Unique vocabulary  : 79,605

Top-20 tokens:
  xxxx                      2,154,562
  the                       1,668,663
  i                         1,491,117
  to                        1,367,610
  and                       1,096,260
  my                         815,067
  a                          802,672
  of                         660,166
  that                       636,204
  was                        541,887
  xx                         536,804
  in                         475,539
  on                         466,688
  they                       461,846
  this                       419,841
  have                       413,753
  not                        403,318
  for                        391,521
  is                         373,624
  me                         368,218

Sample texts (one per category):

[Debt collection]
I paid my debt and when i was getting the amount i was offered a lesser amount to pay and i took the offer and paid and was to

---
### 7. Independence Verification — ANOVA: Text Length vs Category

In [38]:
from scipy.stats import f_oneway

# Does text word-count differ significantly across categories?
# A significant result means length is a discriminative signal (possible leakage).
groups  = [grp['word_length'].values for _, grp in df_sample.groupby('category')]
f_stat, p_val = f_oneway(*groups)

print("=== One-Way ANOVA: Word Length vs Category ===")
print(f"F-statistic : {f_stat:.4f}")
print(f"p-value     : {p_val:.4e}")
if p_val < 0.05:
    print("\n⚠️  Significant: text length varies by category.")
    print("   The model may exploit length as a shortcut — monitor carefully.")
else:
    print("\n✅  Not significant: text length is uniform across categories.")
    print("   Length is not a confounding feature.")

=== One-Way ANOVA: Word Length vs Category ===
F-statistic : 675.1516
p-value     : 0.0000e+00

⚠️  Significant: text length varies by category.
   The model may exploit length as a shortcut — monitor carefully.


---
### 🧠 Core Pedagogical Discussion: Data Quality & Model Expectations

#### What the deduplication step revealed
- The raw dataset contains many **exact (text, label) duplicate pairs** — indicative of templated or batch-submitted tickets.
- Removing them **prevents data leakage** between train/val splits and forces the model to learn from genuinely distinct linguistic patterns.

#### Why 200,000 rows is the right target
| Consideration | Rationale |
|---|---|
| **Statistical power** | 200K rows → ~20K per class (10 classes) — well above the ~5K minimum for robust embedding fine-tuning |
| **Memory budget** | Tokenised sequences (max len 128, int32) ≈ 100 MB — fits in RAM and GPU VRAM |
| **Training speed** | 200K rows trains in minutes per epoch on a modern GPU; 1.9M would take ~9× longer with marginal accuracy gain |
| **Reproducibility** | Smaller, fixed-size datasets are easier to version, share, and benchmark against |
| **Portfolio clarity** | A clearly documented sampling decision demonstrates production-grade engineering judgment |

#### What this means for our Deep Learning models
1. **Achievable Accuracy**: With genuine text signal, a Bi-LSTM + pre-trained embeddings should significantly outperform random chance.
2. **Regularisation**: Even with deduplication, complex models can overfit on minority classes — we will study Dropout, label smoothing, and class-weighted loss.
3. **Production Benchmarking**: Beyond accuracy, we will profile inference latency, CPU/GPU throughput, and memory footprint.